# Dead Time Analysis — Non-Paralyzable Detector
**Method:** Fit the shifted exponential $p(\Delta t) = \lambda\, e^{-\lambda(\Delta t - \tau)}$ for $\Delta t \geq \tau$ to 6 inter-arrival time distributions.  
$\tau$ is **shared** across all runs (detector property); $\lambda_i$ is the **true rate** for run $i$.  
Individual source activities follow as $r_i = \lambda_i - \lambda_{i-1}$.

## The key property of the non-paralyzable model

For a non paralyzable detector with true rate $\lambda$, the inter-arrival time PDF between recorded events is:

$p(t) = \lambda \mathrm{e}^{-\lambda(t - \tau)}, \,\, t \geq \tau$

The crucial point: the exponential slope gives you the true rate $\lambda$ directly -- not the observed (reduced) rate. The date time only shifts the distribution; it does not change the decay constant. This is the most powerful tool.


## Cell 1 — Imports

In [1]:
import ROOT
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

ROOT.gStyle.SetOptStat(111)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

from pathlib import Path
import sys
import ipynbname
project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

# ROOT.gROOT.SetBatch(False)
# ROOT.gStyle.SetOptStat(0)
# ROOT.gStyle.SetOptFit(111)   # show fit params on plot
# ROOT.gStyle.SetPadGridX(True)
# ROOT.gStyle.SetPadGridY(True)
print('ROOT version:', ROOT.gROOT.GetVersion())

from visualization  import hist_quickShow

import ipynbname
import sys
from pathlib import Path
project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

ROOT version: 6.36.06


## Cell 2 — Load data
**Adjust** `FILES` to match your actual filenames.  
If all runs are in **one CSV** with a `run` column, replace the loading block with:
```python
df_all = pd.read_csv('all_runs.csv')
datasets = [df_all[df_all['run']==i]['time_s'].values for i in range(1, 7)]
```

In [2]:
N_RUNS = 6

data_path = project_root/'data/260527'
FILES = [list((data_path).glob(f'time_{i}*.csv'))[0] for i in [29,30,31,32,34,35]]
FILES

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_29_siphraAonly_Timing_Cs-137_1source.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_30_siphraAonly_Timing_Cs-137_2sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_31_siphraAonly_Timing_Cs-137_3sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_32_siphraAonly_Timing_Cs-137_4sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_34_siphraAonly_Timing_Cs-137_5sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_35_siphraAonly_Timing_Cs-137_6sources.csv')]

In [3]:
datasets = []
for i, fname in enumerate(FILES):
    df = pd.read_csv(fname)
    dt = df['time_s'].values.astype(float)
    dt = dt[dt > 0]          # drop any zeroes / artefacts
    datasets.append(dt)

print(f"{'Run':>4}  {'N events':>10}  {'min Δt (s)':>14}  {'mean Δt (s)':>14}  {'1/mean (Hz)':>14}")
print('-' * 62)
for i, dt in enumerate(datasets):
    print(f"{i+1:>4}  {len(dt):>10,}  {dt.min():>14.4e}  {dt.mean():>14.4e}  {1/dt.mean():>14.2f}")

 Run    N events      min Δt (s)     mean Δt (s)     1/mean (Hz)
--------------------------------------------------------------
   1     500,000      1.2000e-05      4.7683e-04         2097.20
   2     500,000      1.2000e-05      3.7966e-04         2633.97
   3     500,000      1.2000e-05      3.0813e-04         3245.43
   4     500,000      1.1000e-05      2.4317e-04         4112.39
   5     500,000      1.2000e-05      1.7851e-04         5601.92
   6     500,000      1.1000e-05      1.7030e-04         5871.83


## Cell 3 — First look: choose histogram binning
Key choices:
- **Lower edge = 0** so the dead-time onset is visible.
- **Bin width** should be small enough to resolve the onset at $\tau$ but not so small that bins are empty in the tail.
- A rule of thumb: aim for ≥ 10 counts/bin in the bulk of the distribution.

In [4]:
# Use the run with the HIGHEST rate to set the time axis
# (it has the shortest mean Δt and will drive the τ estimate)
means = np.array([dt.mean() for dt in datasets])

# Upper edge: 6 × mean of the LOWEST-rate run (run 1)
T_MAX  = 8.0 * means[0]

# Bin width: mean of the HIGHEST-rate run / 40  (empirical, adjust if needed)
BIN_W  = means[-1] / 15.0
N_BINS = int(T_MAX / BIN_W)

print(f'T_MAX  = {T_MAX:.4e} s')
print(f'BIN_W  = {BIN_W:.4e} s')
print(f'N_BINS = {N_BINS}')

T_MAX  = 3.8146e-03 s
BIN_W  = 1.1354e-05 s
N_BINS = 335


## Cell 4 — Build ROOT histograms

In [5]:
# COLORS = [ROOT.kBlue+1, ROOT.kRed+1, ROOT.kGreen+2,
          # ROOT.kMagenta+1, ROOT.kCyan+2, ROOT.kOrange+1]
COLORS = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]


hists = []
for i, dt in enumerate(datasets):
    h = ROOT.TH1D(f'h{i+1}', f'Run {i+1};#Delta t (s);Counts / bin',
                  N_BINS, 0.0, T_MAX)
    # h.SetDirectory(0)      # keep alive outside any TFile
    h.SetLineColor(COLORS[i])
    h.SetLineWidth(1)
    for t in dt:
        if 0 < t <= T_MAX:
            h.Fill(t)
    hists.append(h)
    print(f'Run {i+1}: {h.GetEntries():.0f} entries filled')

Run 1: 499981 entries filled
Run 2: 499981 entries filled
Run 3: 499907 entries filled
Run 4: 499779 entries filled
Run 5: 499771 entries filled
Run 6: 499740 entries filled


## Cell 5 — Visual inspection (log-y)
On a log-y scale a shifted exponential is a straight line starting at $\tau$.  
Look for the **onset** (where counts drop to zero at small $\Delta t$) — that is $\tau$.

In [6]:
if ROOT.gROOT.FindObject('c_inspect'):
    ROOT.gROOT.FindObject('c_inspect').Close()

c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1600, 1200)
c_inspect.Divide(3, 2)
for i, h in enumerate(hists):
    c_inspect.cd(i + 1)
    h.SetLineWidth(2)
    h.Draw('HIST')
    c_inspect.cd(i+1).SetLogy()
c_inspect.Draw()

In [7]:
hist_quickShow(hists[::-1])

(<cppyy.gbl.TCanvas object at 0x642ee29bec40>, None)

In [8]:
# if ROOT.gROOT.FindObject('c_inspect'):
#     ROOT.gROOT.FindObject('c_inspect').Close()

# c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1400, 900)
# c_inspect.Divide(3, 2)
# for i, h in enumerate(hists):
#     c_inspect.cd(i + 1)
#     ROOT.gPad.SetLogy(1)
#     h.Draw('HIST')
# c_inspect.Draw()

## Cell 6 — Initial τ estimate and preliminary λᵢ
The global minimum of all inter-arrival times is a **lower bound** on $\tau$ (the true value is slightly higher due to finite statistics). We use it as a starting point.

For each run we get a rough $\lambda_i$ from $1 / (\bar{\Delta t}_i - \hat{\tau})$.

In [10]:
T_SKIP = 190e-6   # 50 µs — salta la región de pileup (~20 µs) con margen

# Trabajar solo con eventos por encima del corte
datasets_clean = [dt[dt > T_SKIP] for dt in datasets]

ns    = np.array([len(dt)   for dt in datasets_clean])
mins  = np.array([dt.min()  for dt in datasets_clean])
means = np.array([dt.mean() for dt in datasets_clean])

# Estima τ para cada run: posición del bin máximo por encima de T_SKIP
# (para una exponencial desplazada, el modo = τ)
tau_estimates = []
for dt in datasets_clean:
    t_max_hist = np.percentile(dt, 99)
    counts, edges = np.histogram(dt, bins=300, range=(T_SKIP, t_max_hist))
    centers = (edges[:-1] + edges[1:]) / 2
    tau_est = centers[np.argmax(counts)]
    tau_estimates.append(tau_est)

tau_estimates = np.array(tau_estimates)

# Todos deberían dar τ similar (misma propiedad del detector)
# Usamos la media como valor inicial y el mínimo tras el corte como cota superior
TAU_INIT = np.mean(tau_estimates)
TAU_MAX  = mins.min()    # mínimo real tras excluir pileup

print(f"τ por run (mod del histograma): {tau_estimates*1e6}")
print(f"TAU_INIT = {TAU_INIT*1e6:.1f} µs")
print(f"TAU_MAX  = {TAU_MAX*1e6:.1f} µs")

# Ya no usamos TAU_MAX como cota — los eventos en la zona prohibida
# (pileup) invalidan ese argumento

# Bounds para el optimizador basados en los estimados del modo
TAU_LO = 0.5 * TAU_INIT    # cota inferior conservadora
TAU_HI = 2.0 * TAU_INIT    # cota superior conservadora

print(f"Bounds para τ: [{TAU_LO*1e6:.1f}, {TAU_HI*1e6:.1f}] µs")

lam_init = 1.0 / (means - TAU_INIT)
print()
print(f"{'Run':>4}  {'λ_init (Hz)':>14}")
for i, lam in enumerate(lam_init):
    print(f'{i+1:>4}  {lam:>14.4f}')

τ por run (mod del histograma): [202.14166667 204.58       202.00833333 193.59166667 193.66666667
 192.54      ]
TAU_INIT = 198.1 µs
TAU_MAX  = 191.0 µs
Bounds para τ: [99.0, 396.2] µs

 Run     λ_init (Hz)
   1       3255.8305
   2       4881.1809
   3       7221.1257
   4      11575.9776
   5      20581.3316
   6      19252.3631


In [11]:
# Create new set of histograms for the clean data
# update with new deadtime estimation TAU_INIT
datasets_clean = [d[d>=TAU_INIT] for d in datasets_clean]
hists_original = hists
hists = []

for i, dt in enumerate(datasets_clean):
    h = ROOT.TH1D(f'h{i+1}_clean', f'Run {i+1};#Delta t (s);Counts / bin',
                  N_BINS, 0.0, T_MAX)
    # h.SetDirectory(0)      # keep alive outside any TFile
    h.SetLineColor(COLORS[i])
    h.SetLineWidth(1)
    for t in dt:
        if 0 < t <= T_MAX:
            h.Fill(t)
    hists.append(h)
    print(f'Run {i+1}: {h.GetEntries():.0f} entries filled')

Run 1: 449606 entries filled
Run 2: 437694 entries filled
Run 3: 404882 entries filled
Run 4: 334072 entries filled
Run 5: 173944 entries filled
Run 6: 140326 entries filled


In [12]:
if ROOT.gROOT.FindObject('c_inspect'):
    ROOT.gROOT.FindObject('c_inspect').Close()

c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1600, 1200)
c_inspect.Divide(3, 2)
for i, h in enumerate(hists):
    c_inspect.cd(i + 1)
    h.Draw('HIST')
    c_inspect.cd(i+1).SetLogy()
c_inspect.Draw()

## Cell 7 — Individual ROOT fits (diagnostic)
Fit each histogram independently to $A\,e^{-\lambda(\Delta t - \tau)}$ to get per-run estimates of $\lambda_i$ and $\tau$.  
The spread in the fitted $\tau$ values tells you how consistent the detector model is.

> **Note on fit range:** we start the fit at `FIT_START = TAU_INIT * 1.05` to stay safely above the onset and avoid the partial first bin.

In [23]:
# Limpiar canvas y funciones anteriores para evitar conflictos de nombre
for name in ['c_indiv'] + [f'f_ind{i+1}' for i in range(N_RUNS)]:
    obj = ROOT.gROOT.FindObject(name)
    if obj:
        obj.Delete()


TAU_FROM_LED = 198e-6        # µs, del ajuste de eficiencia del LED
FIT_START    = TAU_FROM_LED * 1.10   # 211 µs — claramente sobre el onset
                                      # y sobre cualquier evento espurio

TAU_FIXED = 198e-6

ind_lambdas = []
ind_taus    = []
ind_errs_l  = []
ind_errs_t  = []
fit_funcs   = []

T_FIT_MAXs = [3e-3, 2.3e-3, 1.7e-3, 1e-3, 0.4e-3, 0.3e-3]
# T_FIT_MAXs = [3e-3, 2.3e-3, 1.7e-3, 1e-3, 1e-3, 1e-3]

c_indiv = ROOT.TCanvas('c_indiv', 'Individual fits', 1600, 1200)
c_indiv.Divide(3, 2)

for i, h in enumerate(hists):
    c_indiv.cd(i + 1)
    # ROOT.gPad.SetLogy(1)
    c_indiv.cd(i + 1).SetLogy()

    f = ROOT.TF1(f'f_ind{i+1}',
                 '[0] * exp( -[1] * (x - [2]) )',
                 FIT_START, T_FIT_MAXs[i])
    f.SetParNames('A', '#lambda', '#tau')

    # Initial values
    f.SetParameter(0, h.GetBinContent(h.FindBin(FIT_START * 1.1)))
    f.SetParameter(1, lam_init[i])
    f.SetParameter(2, TAU_FIXED)
    f.FixParameter(2, TAU_FIXED)

    # Bounds
    f.SetParLimits(0, 1e-3, 1e12)
    f.SetParLimits(1, 0.0,  1e12)
    f.SetParLimits(2, TAU_LO,  TAU_HI)

    h.Draw('HIST')
    # 'L' = log-likelihood fit (better for low-count bins), 'R' = use TF1 range
    h.Fit(f'f_ind{i+1}', 'LRQ')   # Q = quiet
    f.SetLineColor(ROOT.kBlack)
    f.SetLineWidth(2)
    f.Draw('SAME')
    fit_funcs.append(f)

    lam = f.GetParameter(1);  el = f.GetParError(1)
    tau = f.GetParameter(2);  et = f.GetParError(2)
    ind_lambdas.append(lam);  ind_errs_l.append(el)
    ind_taus.append(tau);     ind_errs_t.append(et)

c_indiv.Draw()

print(f"{'Run':>4}  {'λ (Hz)':>14}  {'δλ':>10}  {'τ (s)':>14}  {'δτ':>12}")
print('-' * 62)
for i in range(N_RUNS):
    print(f"{i+1:>4}  {ind_lambdas[i]:>14.4f}  {ind_errs_l[i]:>10.4f}"
          f"  {ind_taus[i]:>14.6e}  {ind_errs_t[i]:>12.4e}")

print(f"\nMean τ (individual fits) = {np.mean(ind_taus):.6e} s")
print(f"Std  τ (individual fits) = {np.std(ind_taus):.2e} s")

 Run          λ (Hz)          δλ           τ (s)            δτ
--------------------------------------------------------------
   1       3190.6076      4.9146    2.214690e-04    9.7953e-10
   2       4747.6849      7.4923    2.459120e-04    4.6610e-06
   3       7001.7618     11.7183    2.421650e-04    1.2611e-08
   4      11601.0941     22.4509    2.242306e-04    1.1389e-09
   5      22319.5424     86.0257    1.690119e-04    1.9682e-06
   6      27205.1898    198.3513    1.665892e-04    2.0823e-06

Mean τ (individual fits) = 2.115629e-04 s
Std  τ (individual fits) = 3.22e-05 s


Warning in <TCanvas::Delete()>: may not use this method
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_indiv


In [24]:
# Estimate from exponential shape only

tau_estimates = []
for i, dt in enumerate(datasets_clean):

    t_hi = T_FIT_MAXs[i]
    t_lo = min(TAU_INIT * 1.5, t_hi * 0.5)  # nunca supera la mitad del rango

    counts, edges = np.histogram(dt, bins=200, range=(T_SKIP, T_MAX))
    centers = (edges[:-1] + edges[1:]) / 2

    mask = (centers > t_lo) & (centers < t_hi) & (counts > 0)

    if mask.sum() < 3:
        print(f"Run {i+1}: rango limpio insuficiente "
              f"(t_lo={t_lo*1e6:.0f} µs, t_hi={t_hi*1e6:.0f} µs, "
              f"bins={mask.sum()})")
        tau_estimates.append(np.nan)
        continue

    coeffs    = np.polyfit(centers[mask], np.log(counts[mask]), 1)
    lambda_est = -coeffs[0]

    dt_range  = dt[(dt > T_SKIP) & (dt < t_hi)]
    mean_clean = dt_range.mean()

    tau_est   = mean_clean - 1.0 / lambda_est
    tau_estimates.append(tau_est)
    print(f"Run {i+1}: λ = {lambda_est:.1f} Hz,  τ = {tau_est*1e6:.1f} µs  "
          f"({mask.sum()} bins usados)")

tau_arr = np.array([t for t in tau_estimates if not np.isnan(t)])
print(f"\nMedia τ = {tau_arr.mean()*1e6:.1f} µs,  std = {tau_arr.std()*1e6:.1f} µs")

Run 1: λ = 3094.5 Hz,  τ = 188.9 µs  (149 bins usados)
Run 2: λ = 4657.6 Hz,  τ = 195.4 µs  (110 bins usados)
Run 3: λ = 6911.8 Hz,  τ = 197.1 µs  (77 bins usados)
Run 4: λ = 10573.2 Hz,  τ = 189.5 µs  (39 bins usados)
Run 5: λ = 18916.3 Hz,  τ = 182.1 µs  (11 bins usados)
Run 6: λ = 26392.1 Hz,  τ = 185.6 µs  (6 bins usados)

Media τ = 189.8 µs,  std = 5.2 µs


## Cell 8 — Global fit with shared τ
**Strategy:** minimise a combined binned Poisson negative log-likelihood over all 6 histograms simultaneously, with $\tau$ shared and $\lambda_i$ individual.

For bin $j$ of run $i$ with edges $[a, b]$, the expected count (for $a \geq \tau$) is:
$$\mu_{ij} = n^\text{fit}_i \left[e^{-\lambda_i(a-\tau)} - e^{-\lambda_i(b-\tau)}\right]$$

where $n^\text{fit}_i$ is the number of events **within the fit range** $[\text{FIT\_START},\ T^i_\text{FIT\_MAX}]$. Only bins inside the fit range contribute to the NLL — the tail and sub-onset regions are excluded.


In [20]:
# Events inside each run's fit range — used for normalisation
ns_fit = np.array([
    np.sum((dt > FIT_START) & (dt <= T_FIT_MAXs[i]))
    for i, dt in enumerate(datasets_clean)
])
print("Events per run in fit range:")
for i, n in enumerate(ns_fit):
    print(f"  Run {i+1}: {n:,}")

# Pre-extract bin edges and counts from the clean histograms (once, outside NLL)
bin_edges  = []
bin_counts = []
for h in hists:
    nb_h   = h.GetNbinsX()
    edges  = np.array([h.GetBinLowEdge(b) for b in range(1, nb_h + 2)])
    counts = np.array([h.GetBinContent(b)  for b in range(1, nb_h + 1)])
    bin_edges.append(edges)
    bin_counts.append(counts)


def global_nll(params):
    """
    Combined Poisson NLL over all N_RUNS histograms.
    params = [tau, lambda_1, ..., lambda_N_RUNS]
    Only bins inside [FIT_START, T_FIT_MAXs[i]] contribute.
    """
    tau  = params[0]
    lams = params[1:]

    if tau <= TAU_LO or tau >= TAU_HI:
        return 1e30
    if np.any(lams <= 0):
        return 1e30

    nll = 0.0
    for i in range(N_RUNS):
        a = bin_edges[i][:-1]
        b = bin_edges[i][1:]
        obs = bin_counts[i]

        # Select bins that overlap the fit range [FIT_START, T_FIT_MAX]
        in_range = (b > tau) & (a < T_FIT_MAXs[i])

        a_r   = a[in_range]
        b_r   = b[in_range]
        obs_r = obs[in_range]

        a_eff = np.maximum(a_r, tau)   # clamp left edge to tau
        mu = ns_fit[i] * (np.exp(-lams[i] * (a_eff - tau))
                        - np.exp(-lams[i] * (b_r   - tau)))
        mu = np.maximum(mu, 1e-300)

        nll += np.sum(mu - obs_r * np.log(mu))

    return nll


# ── Initial parameters and bounds ────────────────────────────────────────────
p0     = np.concatenate([[np.mean(ind_taus)], ind_lambdas])
bounds = [(TAU_LO, TAU_HI)] + [(1.0, None)] * N_RUNS

result = minimize(global_nll, p0, method='L-BFGS-B', bounds=bounds,
                  options={'ftol': 1e-12, 'gtol': 1e-8, 'maxiter': 5000})

print()
print('Minimisation status:', result.message)
print(f'Final NLL = {result.fun:.4f}')
print()

tau_fit  = result.x[0]
lams_fit = result.x[1:]

print(f"Global τ  = {tau_fit*1e6:.2f} µs   (individual mean = {np.mean(ind_taus)*1e6:.2f} µs)")
print()
print(f"{'Run':>4}  {'λ_global (Hz)':>16}  {'λ_indiv (Hz)':>16}  {'Δ (%)':>8}")
print('-' * 52)
for i in range(N_RUNS):
    delta = 100 * (lams_fit[i] - ind_lambdas[i]) / ind_lambdas[i]
    print(f"{i+1:>4}  {lams_fit[i]:>16.4f}  {ind_lambdas[i]:>16.4f}  {delta:>8.2f}")


Events per run in fit range:
  Run 1: 434,550
  Run 2: 416,871
  Run 3: 375,960
  Run 4: 291,976
  Run 5: 114,929
  Run 6: 75,115

Minimisation status: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Final NLL = -19853721.4508

Global τ  = 182.19 µs   (individual mean = 237.15 µs)

 Run     λ_global (Hz)      λ_indiv (Hz)     Δ (%)
----------------------------------------------------
   1         3191.6747         3191.6637      0.00
   2         4745.2672         4745.2562      0.00
   3         7019.2575         7019.2485      0.00
   4        11710.9259        11710.9204      0.00
   5        25578.4633        25578.4633      0.00
   6        34163.4984        34163.4984      0.00


In [68]:
TAU_FIXED = 198e-6    # valor del LED, en segundos

def global_nll_fixed_tau(lams):
    """NLL con tau fijado al valor del LED."""
    if np.any(lams <= 0):
        return 1e30
    nll = 0.0
    for i in range(N_RUNS):
        a = bin_edges[i][:-1];  b = bin_edges[i][1:]
        obs = bin_counts[i]
        in_range = (b > TAU_FIXED) & (a < T_FIT_MAXs[i])
        a_eff = np.maximum(a[in_range], TAU_FIXED)
        mu = ns_fit[i] * (np.exp(-lams[i]*(a_eff - TAU_FIXED))
                        - np.exp(-lams[i]*(b[in_range] - TAU_FIXED)))
        mu = np.maximum(mu, 1e-300)
        nll += np.sum(mu - obs[in_range]*np.log(mu))
    return nll

p0     = np.array(ind_lambdas)
bounds = [(1.0, None)] * N_RUNS

result = minimize(global_nll_fixed_tau, p0, method='L-BFGS-B', bounds=bounds,
                  options={'ftol': 1e-12, 'gtol': 1e-8, 'maxiter': 5000})

lams_fit = result.x
tau_fit  = TAU_FIXED

print(f'τ (fijo, del LED) = {tau_fit*1e6:.2f} µs')
for i, lam in enumerate(lams_fit):
    print(f'Run {i+1}: λ = {lam:.2f} Hz')

τ (fijo, del LED) = 198.00 µs
Run 1: λ = 3203.25 Hz
Run 2: λ = 4763.79 Hz
Run 3: λ = 7048.97 Hz
Run 4: λ = 11948.30 Hz
Run 5: λ = 28870.86 Hz
Run 6: λ = 41317.45 Hz


## Cell 9 — Uncertainty on τ via profile likelihood scan
We scan $\tau$ around the best-fit value and find the interval where $\Delta(-2\ln\mathcal{L}) < 1$ (68% CL, 1σ).  
At each fixed $\tau$, all $\lambda_i$ are re-optimised.


In [69]:
# ── Profile scan ─────────────────────────────────────────────────────────────
N_SCAN   = 200
tau_lo_s = tau_fit * 0.70
tau_hi_s = min(TAU_HI * 0.999, tau_fit * 1.30)
tau_scan = np.linspace(tau_lo_s, tau_hi_s, N_SCAN)

nll_scan  = []
lams_scan = []

for tau_s in tau_scan:
    # Warm-start: analytic estimate of optimal λᵢ given τ
    lam_warm = 1.0 / np.maximum(means - tau_s, 1e-12)
    p_s      = np.concatenate([[tau_s], lam_warm])
    # Fix tau by tightening its bounds to a tiny window
    bounds_s = [(tau_s - 1e-15, tau_s + 1e-15)] + [(1.0, None)] * N_RUNS
    res_s = minimize(global_nll, p_s, method='L-BFGS-B', bounds=bounds_s,
                     options={'ftol': 1e-12, 'maxiter': 2000})
    nll_scan.append(res_s.fun)
    lams_scan.append(res_s.x[1:])

nll_scan  = np.array(nll_scan)
delta_nll = 2.0 * (nll_scan - result.fun)   # −2 Δ ln L

# 1σ interval: where Δ(−2 ln L) < 1
mask_1sig   = delta_nll < 1.0
tau_1sig_lo = tau_scan[mask_1sig].min() if mask_1sig.any() else tau_fit
tau_1sig_hi = tau_scan[mask_1sig].max() if mask_1sig.any() else tau_fit
TAU_FROM_LED = 192e-6        # µs, del ajuste de eficiencia del LED
FIT_START    = TAU_FROM_LED * 1.10   # 211 µs — claramente sobre el onset
                                      # y sobre cualquier evento espurio
print(f"τ = {tau_fit*1e6:.2f} µs   [{tau_1sig_lo*1e6:.2f}, {tau_1sig_hi*1e6:.2f}] µs   (1σ)")

# ── Plot profile ──────────────────────────────────────────────────────────────
for name in ['c_profile']:
    obj = ROOT.gROOT.FindObject(name)
    if obj: obj.Delete()

c_profile = ROOT.TCanvas('c_profile', 'Profile likelihood for τ', 800, 500)

g_prof = ROOT.TGraph(int(N_SCAN),
                     (tau_scan * 1e6).astype('d'),
                     delta_nll.astype('d'))
g_prof.SetTitle(';#tau (#mus);-2#Delta ln#it{L}')
g_prof.SetLineWidth(2)
g_prof.SetLineColor(ROOT.kBlue + 1)
g_prof.Draw('AL')

# 1σ line
line_1sig = ROOT.TLine(tau_lo_s * 1e6, 1.0, tau_hi_s * 1e6, 1.0)
line_1sig.SetLineStyle(2); line_1sig.SetLineColor(ROOT.kRed); line_1sig.SetLineWidth(2)
line_1sig.Draw()

# Best-fit marker
line_best = ROOT.TLine(tau_fit * 1e6, 0.0, tau_fit * 1e6, 5.0)
line_best.SetLineStyle(3); line_best.SetLineColor(ROOT.kGray + 1); line_best.SetLineWidth(2)
line_best.Draw()

c_profile.Draw()


τ = 198.00 µs   [138.60, 195.91] µs   (1σ)


Warning in <TCanvas::Delete()>: may not use this method
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_profile


## Cell 10 — Final overlay: data + global fit
Each histogram is drawn with the best-fit curve evaluated at the globally determined $\tau$ and per-run $\lambda_i$.  
A dashed red line marks the onset at $\tau$.


In [70]:
for name in ['c_final'] + [f'f_global{i+1}' for i in range(N_RUNS)]:
    obj = ROOT.gROOT.FindObject(name)
    if obj: obj.Delete()

c_final = ROOT.TCanvas('c_final', 'Global fit — all runs', 1600, 1200)
c_final.Divide(3, 2)

# Keep references alive (ROOT GC protection)
_fit_objects = []

for i, h in enumerate(hists):
    c_final.cd(i + 1)
    c_final.cd(i + 1).SetLogy()

    # Amplitude: match the model to the first populated bin above FIT_START
    first_bin = h.FindBin(FIT_START * 1.01)
    while h.GetBinContent(first_bin) == 0 and first_bin < h.GetNbinsX():
        first_bin += 1
    t_first = h.GetBinCenter(first_bin)
    A_norm  = h.GetBinContent(first_bin) * np.exp(lams_fit[i] * (t_first - tau_fit))

    f = ROOT.TF1(f'f_global{i+1}',
                 '[0] * exp( -[1] * (x - [2]) )',
                 tau_fit, T_FIT_MAXs[i])
    f.SetParameter(0, A_norm)
    f.SetParameter(1, lams_fit[i])
    f.SetParameter(2, tau_fit)
    f.SetLineColor(ROOT.kBlack); f.SetLineWidth(2)

    h.SetTitle(f'Run {i+1}  (#lambda = {lams_fit[i]:.0f} Hz, #tau = {tau_fit*1e6:.1f} #mus)'
               f';#Delta t (s);Counts / bin')
    h.Draw('HIST')
    f.Draw('SAME')

    # τ onset marker
    line_tau = ROOT.TLine(tau_fit, h.GetMinimum(1), tau_fit, h.GetMaximum())
    line_tau.SetLineStyle(2); line_tau.SetLineColor(ROOT.kRed); line_tau.SetLineWidth(2)
    line_tau.Draw()

    _fit_objects.append((f, line_tau))   # keep alive

c_final.Draw()


Warning in <TCanvas::Delete()>: may not use this method
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_final


## Cell 11 — Source activities
$r_i = \lambda_i - \lambda_{i-1}$ is the activity of the $i$-th source added.  
Uncertainties propagate from the Cramér-Rao bound $\text{Var}(\hat{\lambda}_i) \approx \lambda_i^2 / n^\text{fit}_i$.


In [71]:
# Variances from Cramér–Rao bound for shifted exponential (fixed τ)
# Var(λ̂ᵢ) = λᵢ² / n_fit_i
lam_vars = lams_fit**2 / ns_fit
lam_errs = np.sqrt(lam_vars)

activities    = np.zeros(N_RUNS)
activity_errs = np.zeros(N_RUNS)

activities[0]    = lams_fit[0]
activity_errs[0] = lam_errs[0]

for i in range(1, N_RUNS):
    activities[i]    = lams_fit[i] - lams_fit[i - 1]
    activity_errs[i] = np.sqrt(lam_vars[i] + lam_vars[i - 1])

print('=' * 70)
print(f"  Dead time  τ = {tau_fit*1e6:.2f} µs"
      f"   [{tau_1sig_lo*1e6:.2f}, {tau_1sig_hi*1e6:.2f}] µs   (1σ profile)")
print('=' * 70)
print(f"{'Source':>8}  {'λᵢ (Hz)':>12}  {'δλᵢ':>10}  {'rᵢ (Hz)':>12}  {'δrᵢ':>10}")
print('-' * 58)
for i in range(N_RUNS):
    print(f"{i+1:>8}  {lams_fit[i]:>12.2f}  {lam_errs[i]:>10.2f}"
          f"  {activities[i]:>12.2f}  {activity_errs[i]:>10.2f}")
print('=' * 70)

# Sanity check
if np.all(activities > 0):
    print('\n✓ All source activities are positive — fit is self-consistent.')
else:
    neg = [i+1 for i in range(N_RUNS) if activities[i] <= 0]
    print(f'\n⚠  Negative activity for source(s) {neg} — check fit range or data.')

# Dead-time fraction at each run's observed rate
print()
print(f"{'Run':>4}  {'m_obs (Hz)':>12}  {'f_dead (%)':>12}  {'n_true (Hz)':>14}")
print('-' * 48)
for i in range(N_RUNS):
    m_obs  = ns_fit[i] / datasets_clean[i].shape[0] * len(datasets_clean[i]) /              (np.sum(datasets_clean[i] <= T_FIT_MAXs[i]) / ns_fit[i] * 
              (datasets_clean[i].max()))
    # simpler: use lams_fit and tau_fit directly
    n_true  = lams_fit[i]
    f_dead  = n_true * tau_fit / (1 + n_true * tau_fit) * 100
    m_obs   = n_true / (1 + n_true * tau_fit)
    print(f"{i+1:>4}  {m_obs:>12.2f}  {f_dead:>12.2f}  {n_true:>14.2f}")


  Dead time  τ = 198.00 µs   [138.60, 195.91] µs   (1σ profile)
  Source       λᵢ (Hz)         δλᵢ       rᵢ (Hz)         δrᵢ
----------------------------------------------------------
       1       3203.25        4.86       3203.25        4.86
       2       4763.79        7.38       1560.53        8.83
       3       7048.97       11.50       2285.18       13.66
       4      11948.30       22.11       4899.33       24.92
       5      28870.86       85.16      16922.56       87.99
       6      41317.45      150.75      12446.59      173.15

✓ All source activities are positive — fit is self-consistent.

 Run    m_obs (Hz)    f_dead (%)     n_true (Hz)
------------------------------------------------
   1       1960.08         38.81         3203.25
   2       2451.48         48.54         4763.79
   3       2942.35         58.26         7048.97
   4       3549.95         70.29        11948.30
   5       4298.54         85.11        28870.86
   6       4500.39         89.11        41

## Cell 12 — Summary bar chart of activities

In [72]:
for name in ['c_bar']:
    obj = ROOT.gROOT.FindObject(name)
    if obj: obj.Delete()

c_bar = ROOT.TCanvas('c_bar', 'Source activities', 800, 500)

x_vals = np.arange(1, N_RUNS + 1, dtype='d')
y_vals = activities.astype('d')
ex     = np.zeros(N_RUNS, dtype='d')
ey     = activity_errs.astype('d')

g_act = ROOT.TGraphErrors(int(N_RUNS), x_vals, y_vals, ex, ey)
g_act.SetTitle('Individual source activities;Source index;Activity (Hz)')
g_act.SetMarkerStyle(20)
g_act.SetMarkerSize(1.5)
g_act.SetMarkerColor(ROOT.kBlue + 1)
g_act.SetLineColor(ROOT.kBlue + 1)
g_act.SetLineWidth(2)
g_act.Draw('AP')
c_bar.Draw()


Warning in <TCanvas::Delete()>: may not use this method
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_bar


### Quantify uncertainty in the last acquisitions

In [73]:
tau_values = [189e-6, 192e-6, 198e-6]   # rango del LED
lams_per_tau = {}

for tau_test in tau_values:
    def nll_tau(lams):
        if np.any(lams <= 0): return 1e30
        nll = 0.0
        for i in range(N_RUNS):
            a = bin_edges[i][:-1]; b = bin_edges[i][1:]
            obs = bin_counts[i]
            in_range = (b > tau_test) & (a < T_FIT_MAXs[i])
            a_eff = np.maximum(a[in_range], tau_test)
            mu = ns_fit[i] * (np.exp(-lams[i]*(a_eff - tau_test))
                            - np.exp(-lams[i]*(b[in_range] - tau_test)))
            mu = np.maximum(mu, 1e-300)
            nll += np.sum(mu - obs[in_range]*np.log(mu))
        return nll

    res = minimize(nll_tau, np.array(ind_lambdas), method='L-BFGS-B',
                   bounds=[(1.0, None)]*N_RUNS,
                   options={'ftol':1e-12, 'maxiter':5000})
    lams_per_tau[tau_test] = res.x

# Sistemática = mitad del rango (max - min)
lams_array = np.array([lams_per_tau[t] for t in tau_values])
delta_syst  = (lams_array.max(axis=0) - lams_array.min(axis=0)) / 2

print(f"{'Run':>4}  {'λ (Hz)':>10}  {'δstat':>8}  {'δsyst':>8}  {'δtotal':>8}")
print('-' * 46)
for i in range(N_RUNS):
    d_stat  = lam_errs[i]
    d_syst  = delta_syst[i]
    d_total = np.sqrt(d_stat**2 + d_syst**2)
    print(f"{i+1:>4}  {lams_fit[i]:>10.1f}  "
          f"{d_stat:>8.1f}  {d_syst:>8.1f}  {d_total:>8.1f}")
    

 Run      λ (Hz)     δstat     δsyst    δtotal
----------------------------------------------
   1      3203.3       4.9      20.8      21.3
   2      4763.8       7.4      47.0      47.6
   3      7049.0      11.5      82.6      83.4
   4     11948.3      22.1     196.5     197.7
   5     28870.9      85.2     465.5     473.2
   6     41317.5     150.8    2845.9    2849.9
